<a href="https://colab.research.google.com/github/kaveesha82/DS_Project/blob/component-1/arrival_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_excel('/content/drive/MyDrive/DSGP/monthlyarrivals.xlsx')

In [5]:
#Clean column names
df.columns = df.columns.str.strip()

#Remove commas inside numbers
df = df.replace(",", "", regex=True)


df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1452 entries, 0 to 1451
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Year         1452 non-null   int64  
 1   Month        1452 non-null   object 
 2   Nationality  1452 non-null   object 
 3   Arrivals     1452 non-null   int64  
 4   Avg_Stay     1056 non-null   float64
 5   Revenue_USD  1056 non-null   float64
dtypes: float64(2), int64(2), object(2)
memory usage: 68.2+ KB


In [6]:
df.shape


(1452, 6)

In [7]:
df.columns

Index(['Year', 'Month', 'Nationality', 'Arrivals', 'Avg_Stay', 'Revenue_USD'], dtype='object')

In [8]:
df.dtypes

,0
Year,int64
Month,object
Nationality,object
Arrivals,int64
Avg_Stay,float64
Revenue_USD,float64


In [9]:
df.isnull().sum()

,0
Year,0
Month,0
Nationality,0
Arrivals,0
Avg_Stay,396
Revenue_USD,396


In [10]:
#Encode months
month_map = {
    "January":1, "February":2, "March":3, "April":4,
    "May":5, "June":6, "July":7, "August":8,
    "September":9, "October":10, "November":11, "December":12
}

df["Month_Num"] = df["Month"].map(month_map)

In [11]:
df.head()

,Year,Month,Nationality,Arrivals,Avg_Stay,Revenue_USD,Month_Num
0,2025,January,India,45496,6.8,58310242.0,1
1,2025,January,Russian Federation,37914,12.5,89335013.0,1
2,2025,January,United Kingdom,20220,13.2,50311716.0,1
3,2025,January,Germany,17693,14.2,47358358.0,1
4,2025,January,China,15165,8.5,24300108.0,1


In [12]:
df = df.sort_values(["Nationality", "Year", "Month_Num"])

In [13]:
df["Date"] = pd.to_datetime(
    df["Year"].astype(str) + "-" + df["Month_Num"].astype(str) + "-01"
)


In [14]:
df["Month_Name"] = df["Date"].dt.month_name()

seasonality = (
    df.groupby(["Nationality", "Month_Name"])["Arrivals"]
      .mean()
      .reset_index()
)


In [15]:
peak_months = (
    seasonality
    .sort_values(["Nationality", "Arrivals"], ascending=False)
    .groupby("Nationality")
    .first()
)

peak_months


,Month_Name,Arrivals
Nationality,,
Australia,December,9064.272727
Bangladesh,May,7975.000000
Belarus,February,44.000000
Canada,July,3852.166667
China,February,18255.000000
France,February,10789.700000
Germany,February,12495.545455
India,December,36736.181818
Iran,March,3738.000000


In [16]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error
import numpy as np

results_list = []
forecasts = {}

for nat in df["Nationality"].unique():

    series = (
        df[df["Nationality"] == nat]
        .set_index("Date")["Arrivals"]
    )

    # Skip if data is too short
    if len(series) < 24:
        continue

    train = series[:-12]
    test = series[-12:]

    # Holt-Winters
    hw = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=12
    ).fit()

    hw_pred = hw.forecast(12)
    hw_rmse = np.sqrt(mean_squared_error(test, hw_pred))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/

ValueError: Cannot compute initial seasonals using heuristic method with less than two full seasonal cycles in the data.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
